Maybe try to read in the opera datasets into MintPy for improved time series velocities and ascending/descending decomposition? Then clip the datasets by the aquifers for zonal statistics?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mintpy import view, tsview
import os, shutil
from pathlib import Path
import geopandas as gpd
from transboundary_opera.decomposer import InSARDecomposer
import transboundary_opera.decomposition_tools as dt

In [ ]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")


data_storage = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # linux
# data_storage = Path('/mnt/c/Users/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # WSL
mintpy_paths = list(data_storage.glob('*/mintpy'))
opera_product_paths = list(data_storage.glob('*/subset-ncs/*.nc'))

In [ ]:
mintpy_paths

In [ ]:
# overlapping_pairs = get_asc_desc_pairs(data_storage)

overlapping_pairs = dt.get_asc_desc_pairs(data_storage)
print(len(overlapping_pairs))

In [ ]:
overlapping_pairs

In [ ]:
def analyze_geometry_frame_intersections(gdf, frame_ids):
    """
    For each geometry in GDF, find which frames intersect it.
    Uses bounding boxes for API search, then filters by actual intersection.
    
    Parameters
    ----------
    gdf : GeoDataFrame
        Input GeoDataFrame with polygons
    frame_ids : list
        List of all frame IDs to check
    
    Returns
    -------
    dict
        Mapping of {geometry_index: [intersecting_frame_ids]}
    dict
        Mapping of {geometry_index: GeoDataFrame of intersecting frames}
    """
    results = {}
    frame_gdfs = {}
    
    for idx, geom_row in gdf.iterrows():
        # Get frames using simplified bbox (fast API search)
        geom_frames = get_frame_geometries(
            frame_ids, 
            gdf_bounds=geom_row.geometry.bounds  # Use individual geometry bbox
        )
        
        # Filter to only frames that actually intersect the complex geometry
        intersecting_frames = []
        for _, frame_row in geom_frames.iterrows():
            if geom_row.geometry.intersects(frame_row.geometry):
                intersecting_frames.append(frame_row['frame_id'])
        
        results[idx] = intersecting_frames
        # Store the actual intersecting frame geometries
        frame_gdfs[idx] = geom_frames[geom_frames['frame_id'].isin(intersecting_frames)]
    
    return results, frame_gdfs

In [ ]:
frame_ids = dt.get_unique_frame_ids(gdf, track_per_row=True, max_workers=8)
frame_ids

## Process all identified pairs

In [ ]:
processor = InSARDecomposer(overlapping_pairs, ds_name='timeseries', angle=-90)
processor.run()
print(processor.successful_pairs)
print(processor.failed_pairs)

In [ ]:
# TODO: Make sure that we have downloaded all of the data covering the aquifers, spatially
# TODO: Exapnd the number of days we are searching

for pair in processor.successful_pairs:
    fig = dt.plot_displacements(pair['horz'], pair['vert'], time_idx=-1, gdf=gdf)

In [ ]:
# examine outputs to check
test_files = ['/home/clayc/Documents/US_Mex_InSAR/OPERA_data/16939_18905_dhorz.h5', '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/16939_18905_dvert.h5']
fig = dt.plot_displacements(test_files[0], test_files[1], time_idx=-1, gdf=gdf)

## Process individual pairs and visualize them alongside intersecting aquifer boundaries

In [ ]:
pair = overlapping_pairs[2]
processor = InSARDecomposer(pair, ds_name='timeseries', azimuth=-90)

outfiles = InSARDecomposer.process_pair(processor, pair=pair)
fig = dt.plot_displacements(outfiles['horz'], outfiles['vert'], time_idx=0, gdf=gdf)